<a href="https://colab.research.google.com/github/tahiyat-rafi/Introduction-to-deep-learning-/blob/main/solving_a_2D_wave_equation_using_PINN_(OOP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import torch
import torch.nn as nn

#pde: u_tt = c**2*(u_xx + u_yy)
#class
class pinn(nn.Module):
  def __init__(self,hidden_dim=32,c = 1.0,lr=.001):
      super().__init__()
      self.c = c

      self.net = nn.Sequential(
      nn.Linear(3,hidden_dim),
      nn.Tanh(),
      nn.Linear(hidden_dim,hidden_dim),
      nn.Tanh(),
      nn.Linear(hidden_dim,hidden_dim),
      nn.Linear(hidden_dim,1)
         )

      #optimizer
      self.optimizer = torch.optim.Adam(self.parameters(),lr=lr)

  def forward(self,x,y,t):
      inputs = torch.cat([x,y,t], dim=1)
      return self.net(inputs)

        #loss function
  def loss(self,x,y,t):
      x.requires_grad=True
      y.requires_grad=True
      t.requires_grad=True
      u = self.forward(x,y,t)
      #df
      u_x = torch.autograd.grad(u,x,grad_outputs=torch.ones_like(u),create_graph=True)[0]
      u_y = torch.autograd.grad(u,y,grad_outputs=torch.ones_like(u),create_graph=True)[0]
      u_t = torch.autograd.grad(u,t,grad_outputs=torch.ones_like(u),create_graph=True)[0]
      #ddf
      u_xx = torch.autograd.grad(u_x,x,grad_outputs=torch.ones_like(u_x),create_graph=True)[0]
      u_yy = torch.autograd.grad(u_y,y,grad_outputs=torch.ones_like(u_y),create_graph=True)[0]
      u_tt = torch.autograd.grad(u_t,t,grad_outputs=torch.ones_like(u_t),create_graph=True)[0]
      err = u_tt-(self.c**2)*(u_xx+u_yy)
      return torch.mean(err**2)

        #training
  def train(self,epochs=1000,points=500):
      for epoch in range(epochs):
        self.optimizer.zero_grad()
        x = torch.rand(points,1)*2-1
        y = torch.rand(points,1)*2-1
        t = torch.rand(points,1)*1

        loss_val = self.loss(x,y,t)
        loss_val.backward()
        self.optimizer.step()

        if epoch % 200 == 0:
          print(f"epoch: {epoch:4d}, loss: {loss_val.item():.6f}")
      print("Training Completed\n")
#objects
if __name__ == "__main__":
  print("\n First model 64 hidden layes\n")
  model1=pinn(hidden_dim=64,c=2.3,lr=.002)
  model1.train(epochs=1000,points=2000)

  print("Second model 32 hidden layers\n")
  model2=pinn(hidden_dim=32,c=2.3,lr=.002)
  model2.train(epochs=1000,points=2000)




 First model 64 hidden layes

epoch:    0, loss: 0.019385
epoch:  200, loss: 0.000002
epoch:  400, loss: 0.000001
epoch:  600, loss: 0.000000
epoch:  800, loss: 0.000000
Training Completed

Second model 32 hidden layers

epoch:    0, loss: 0.021867
epoch:  200, loss: 0.000006
epoch:  400, loss: 0.000002
epoch:  600, loss: 0.000001
epoch:  800, loss: 0.000000
Training Completed

